---
# Clinical Data Quality Engine (DQIE)
# Notebook 06 — Dashboard Export
# Purpose: Prepare final Gold Layer datasets for dashboards
---

# Preparations
---

## Environment Setup

In [ ]:
import sys
import os
import json
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PYTHONPATH OK:", PROJECT_ROOT)

# Load Gold Layer — Scoring Outputs
---

In [ ]:
entity_scores = pd.read_parquet("../data/_3_gold/scoring/entity_scores.parquet")

with open("../data/_3_gold/scoring/dataset_score.json", "r") as f:
    dataset_score = json.load(f)

print("Gold Layer scoring outputs loaded.")
display(entity_scores.head())
dataset_score

# Prepare Dashboard Tables
---

## 1. Entity Scores Table 
Already ready - used for:
- patient scores
- injury scores
- session scores
- report scores
- severity distribution
- DQI by entity type

## 2. Dataset Score Table (JSON → DataFrame)

In [ ]:
dataset_score_df = pd.DataFrame([{
    "global_score": dataset_score["global_score"],
    "global_label": dataset_score["global_label"],
    "weighted_global_score": dataset_score["weighted_global_score"],
    "weighted_global_label": dataset_score["weighted_global_label"],
    "anomaly_density": dataset_score["anomaly_density"],
    "multi_source_entities": dataset_score["sources_reliability"]["multi_source_entities"],
    "percentage_multi_source": dataset_score["sources_reliability"]["percentage_multi_source"]
}])

display(dataset_score_df)

## 3. Severity Distribution Table

In [ ]:
severity_dist_df = pd.DataFrame(
    list(dataset_score.get("severity_distribution", {}).items()),
    columns=["severity_level", "count"]
)

display(severity_dist_df)

## 4. Score by Entity Type Table

In [ ]:
score_by_type_df = pd.DataFrame(
    list(dataset_score.get("score_by_entity_type", {}).items()),
    columns=["entity_type", "avg_score"]
)

display(score_by_type_df)

## 5. Entity-Type Contribution Table

In [ ]:
entity_type_contrib_df = pd.DataFrame(
    list(dataset_score.get("entity_type_contribution", {}).items()),
    columns=["entity_type", "weighted_score"]
)

display(entity_type_contrib_df)

# Save Dashboard Tables to Gold Layer
---

In [ ]:
output_dir = "../data/_3_gold/dashboard/"
os.makedirs(output_dir, exist_ok=True)

entity_scores.to_parquet(f"{output_dir}/entity_scores.parquet", index=False)
dataset_score_df.to_parquet(f"{output_dir}/dataset_score.parquet", index=False)
severity_dist_df.to_parquet(f"{output_dir}/severity_distribution.parquet", index=False)
score_by_type_df.to_parquet(f"{output_dir}/score_by_entity_type.parquet", index=False)
entity_type_contrib_df.to_parquet(f"{output_dir}/entity_type_contribution.parquet", index=False)

print("Dashboard tables saved to Gold Layer.")

# Summary
---

In [ ]:
print("\n--- DASHBOARD EXPORT SUMMARY ---")
print(f"Entity scores rows: {len(entity_scores)}")
print(f"Dataset score rows: {len(dataset_score_df)}")
print(f"Severity distribution rows: {len(severity_dist_df)}")
print(f"Score by entity type rows: {len(score_by_type_df)}")
print(f"Entity-type contribution rows: {len(entity_type_contrib_df)}")

print("\nDashboard export completed. Ready for Power BI / Streamlit / Databricks.")